In [1]:
import sys
import os
import json
import re

# Fix import path
sys.path.append(os.path.abspath("../"))

texts = []
sources = []

with open("../data/processed/class8_science.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        texts.append(data["text"])
        sources.append(data["source"])

print("Total chunks:", len(texts))

Total chunks: 1075


In [2]:
def clean_words(text):
    return re.findall(r'\b\w+\b', text.lower())


def improved_search(query, texts, top_k=3):
    query_words = set(clean_words(query))

    keywords = ["photosynthesis", "plant", "chlorophyll", "sunlight"]

    scores = []

    for i, text in enumerate(texts):
        text_words = set(clean_words(text))
        
        # base score
        score = len(query_words.intersection(text_words))
        
        # boost using keywords
        score += sum(2 for k in keywords if k in text.lower())
        
        scores.append((score, i))

    scores.sort(reverse=True)

    top_indices = [idx for score, idx in scores if score > 0][:top_k]

    return top_indices

In [3]:
query = "What is photosynthesis?"

top_indices = improved_search(query, texts)

retrieved_chunks = [texts[i] for i in top_indices]

for i in top_indices:
    print("📖 Source:", sources[i])
    print(texts[i])
    print("-----")

📖 Source: chapter2
. A step further Cells in all parts of a plant have tiny rodshaped structures called plastids. Some plastids, like chloroplasts, contain chlorophyll, which makes them green and helps in photosynthesis. In nongreen parts, they help in the storage of substances. Plant cells also have a large, emptylooking space called a vacuole. This helps the plant cell store important substances, get rid of waste, and maintain the shape of the cell. This gives strength and support to the plant
-----
📖 Source: chapter13
. You also learnt about the important life processes in plants and animals. Now, let us explore how all these elements interact to support and sustain life on Earth. .. Air, water, and sunlight We know that the atmosphere contains oxygen, which humans, animals, and plants use for respiration. In the presence of sunlight, plants take carbon dioxide from the air and water from Reprint  CChhaapptteerr ..iinndddd    PPMM the soil to prepare food by photosynthesis
-----
📖 S

In [7]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3")

In [8]:
def generate_answer(query, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a Class 8 Science tutor.

Answer ONLY from the given context.
Use simple language.

If answer is not in context, say:
"I’m focused on Class 8 Science."

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)
    return response.content

In [9]:
answer = generate_answer(query, retrieved_chunks)

print("🤖 Answer:\n")
print(answer)

🤖 Answer:

Photosynthesis is when plants take carbon dioxide from the air and water from the soil to prepare food, using sunlight!
